# DBNL Rao pipeline — Colab runner

Code comes from GitHub (`trister95/unseen-semantic-diversity`).
Data + secrets come from Drive (`MyDrive/dbnl/`).

Layout on the VM:
- `/content/code/` — clone of the repo (`functional_diversity/rao/...` inside)
- `/content/data/dbnl_txt_files/` — extracted from Drive tar.gz
- `/content/drive/MyDrive/dbnl/` — persistent Drive store (metadata CSV, .env, work backup)
- `/content/work/` — outputs; explicitly copied to Drive after each stage

## 1. Mount Drive, load `.env`, check GPU

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

ENV_PATH = '/content/drive/MyDrive/dbnl/.env'
with open(ENV_PATH) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        k, _, v = line.partition('=')
        os.environ[k.strip()] = v.strip().strip('"').strip("'")
assert 'HF_TOKEN' in os.environ, '.env loaded but no HF_TOKEN found'
print('HF_TOKEN starts with:', os.environ['HF_TOKEN'][:8] + '…')

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Clone (or pull) the repo

Public repo: HTTPS clone, no token needed.
Re-running pulls instead of cloning, so cell is idempotent.

In [ ]:
REPO_URL = 'https://github.com/trister95/unseen-semantic-diversity.git'
REPO_DIR = '/content/code'

import os
if os.path.isdir(REPO_DIR + '/.git'):
    !cd $REPO_DIR && git pull --ff-only
else:
    !git clone $REPO_URL $REPO_DIR

%cd $REPO_DIR
!ls functional_diversity/rao

## 3. Install dependencies

In [ ]:
!pip install -q -U transformers spacy

## 4. Extract data to local VM disk

Reading from `/content/drive/` during the pipeline is much slower than `/content/`. One-time tar extract per session.

In [ ]:
if not os.path.exists('/content/data/dbnl_txt_files'):
    !mkdir -p /content/data
    !tar -xzf /content/drive/MyDrive/dbnl/dbnl_txt_files.tar.gz -C /content/data
print('files:', len(os.listdir('/content/data/dbnl_txt_files')))

## 5. Restore prior work from Drive (resume) + start checkpoint loop

If a previous run was interrupted, copying the DB from Drive lets `build_corpus.py` resume from where it stopped. The background loop pushes the DB back to Drive every 5 min so we don't lose work to a session timeout.

In [ ]:
!mkdir -p /content/work
!cp -n /content/drive/MyDrive/dbnl/work/dbnl.sqlite /content/work/ 2>/dev/null || echo '(no prior DB — fresh run)'
!cp -n /content/drive/MyDrive/dbnl/work/dbnl_empty.sqlite /content/work/ 2>/dev/null || true

import subprocess
_checkpoint = subprocess.Popen(
    "while true; do "
    "  mkdir -p /content/drive/MyDrive/dbnl/work; "
    "  cp /content/work/dbnl.sqlite /content/drive/MyDrive/dbnl/work/ 2>/dev/null; "
    "  cp /content/work/dbnl_empty.sqlite /content/drive/MyDrive/dbnl/work/ 2>/dev/null; "
    "  sleep 300; "
    "done",
    shell=True,
)
print('checkpoint loop pid:', _checkpoint.pid)

## 6. Stage 1 — smoke test (50 files, 1600-1700)

End-to-end check. Expected ~3 min on T4 with sentencizer + cross-file batching.

In [ ]:
# Uncomment to start clean (otherwise resume skips already-processed files)
# !rm -f /content/work/dbnl.sqlite /content/work/dbnl_empty.sqlite

!python functional_diversity/rao/build_corpus.py \
    --input-dir /content/data/dbnl_txt_files \
    --db /content/work/dbnl.sqlite \
    --empty-db /content/work/dbnl_empty.sqlite \
    --metadata /content/drive/MyDrive/dbnl/dbnl_metadata.csv \
    --min-year 1600 --max-year 1700 \
    --limit 50 --log-every 5 --device cuda:0

## 7. Stage 1 — real run

Year range editable. Estimated wall times on T4 @ ~0.29 files/s:
- 1600-1700: ~40 min (673 files)
- 1500-1700: ~50-120 min
- 1500-1800: ~5 h

Resumes via the `docs` table — re-launching after a session timeout picks up where it left off.

In [ ]:
!python functional_diversity/rao/build_corpus.py \
    --input-dir /content/data/dbnl_txt_files \
    --db /content/work/dbnl.sqlite \
    --empty-db /content/work/dbnl_empty.sqlite \
    --metadata /content/drive/MyDrive/dbnl/dbnl_metadata.csv \
    --min-year 1600 --max-year 1700 \
    --log-every 50 --device cuda:0

# Explicit final sync (background loop catches it within 5 min, but be explicit before moving on)
!cp /content/work/dbnl.sqlite /content/drive/MyDrive/dbnl/work/
!cp /content/work/dbnl_empty.sqlite /content/drive/MyDrive/dbnl/work/

## 8. Quick SQLite sanity peek

In [ ]:
import sqlite3
con = sqlite3.connect('/content/work/dbnl.sqlite')
print('docs     :', con.execute('SELECT COUNT(*) FROM docs').fetchone()[0])
print('sentences:', con.execute('SELECT COUNT(*) FROM sentences').fetchone()[0])
print('mentions :', con.execute('SELECT COUNT(*) FROM mentions').fetchone()[0])

bad = con.execute("""
    SELECT COUNT(*) FROM mentions m JOIN sentences s ON m.sentence_id = s.sentence_id
    WHERE m.doc_id != s.doc_id
""").fetchone()[0]
print('orphan mentions (should be 0):', bad)

print('\n--- top 15 animal surface forms ---')
for txt, n in con.execute(
    'SELECT text, COUNT(*) c FROM mentions GROUP BY text ORDER BY c DESC LIMIT 15'
):
    print(f'  {n:6d}  {txt}')
con.close()

## 9. Stage 2 — occurrence embeddings

In [ ]:
!python functional_diversity/rao/embed_mentions.py \
    --db /content/work/dbnl.sqlite \
    --output /content/work/occurrences.npz \
    --device cuda:0

!cp /content/work/occurrences.npz /content/drive/MyDrive/dbnl/work/

import numpy as np
z = np.load('/content/work/occurrences.npz')
print({k: z[k].shape for k in z.files})

## 10. Stage 3 — occurrence-level Rao Q per decade

In [ ]:
!python functional_diversity/rao/occurrence_rao.py \
    --occurrences /content/work/occurrences.npz \
    --metadata /content/drive/MyDrive/dbnl/dbnl_metadata.csv \
    --output /content/work/rao_occurrence.jsonl \
    --plot-output /content/work/rao_occurrence.png

!cp /content/work/rao_occurrence.jsonl /content/drive/MyDrive/dbnl/work/
!cp /content/work/rao_occurrence.png /content/drive/MyDrive/dbnl/work/

print('\n--- JSONL ---')
!cat /content/work/rao_occurrence.jsonl

## 11. Display the plot inline

In [ ]:
from IPython.display import Image
Image('/content/work/rao_occurrence.png')